In [1]:
from pathlib import Path
import sys
import os
import clingo
project_root = "."


In [2]:
from typing import Any

INSTANCE = "tree"

ASPGARP_FILES = [
    f"{project_root}/single/sim.lp",
    f"{project_root}/adapter.lp",
]

REFERENCE_MODEL = [
    f"{project_root}/models/{INSTANCE}/db.lp",
    f"{project_root}/models/{INSTANCE}/ref.lp",
]

OUTPUT = f"{project_root}/out/{INSTANCE}"

#create output directory if it doesn't exist
os.makedirs(OUTPUT, exist_ok=True)

In [3]:
# Find all states from reference model

def states_from_reference_model(RMODEL, ASPGARPFILES) -> list[str]:
    states = []
    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in RMODEL + ASPGARPFILES:
        ctl.load(path)
    ctl.add("base", [], "#show obs/1.")
    ctl.ground([("base", [])])

    for i, m in enumerate(ctl.solve(yield_=True)):
        new_pos_example = f"state(SID,{i}) :- "
        for atom in m.symbols(shown=True):
            if atom.name == "obs" and len(atom.arguments) == 1:
                varname, value, direction  = atom.arguments[0].arguments
                new_pos_example += f"holds(SID,{varname},{value},{direction}), "
        new_pos_example = new_pos_example.rstrip(", ") + "."  # end of the new example
        states.append(new_pos_example)
    return states

In [4]:
# Reuse the above and find all the transitions between them

def transitions_from_reference_model(RMODEL, ASPGARPFILES, state_descriptions) -> list[str]:
    transitions = []
    ctl = clingo.Control(["0", "--project", "--warn=none"])
    for path in REFERENCE_MODEL + ASPGARP_FILES:
        ctl.load(path)
    ctl.add("base", [],  "\n".join(state_descriptions))
    ctl.add("base", [], "#show state/2.")
    ctl.ground([("base", [])])

    for i, m in enumerate(ctl.solve(yield_=True)):
        from_state = None
        to_state = None
        for atom in m.symbols(shown=True):
            if atom.name == "state" and len(atom.arguments) == 2:
                if atom.arguments[0] == clingo.Number(0): 
                    from_state = atom
                else:
                    to_state = atom
        if from_state and to_state:
            transitions.append(f"edge(({from_state.arguments[1]},{to_state.arguments[1]})).")
    return transitions

In [5]:
state_descriptions = states_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES)
transition_descriptions = transitions_from_reference_model(REFERENCE_MODEL, ASPGARP_FILES, state_descriptions)

#write the state and transition descriptions to files
with open(f"{OUTPUT}/transitions.lp", "w") as f:
    f.write("\n".join(transition_descriptions))

with open(f"{OUTPUT}/states.lp", "w") as f:
    f.write("\n".join(state_descriptions))

!clingo --project --warn=none --outf=2 {OUTPUT}/transitions.lp  | clingraph --out=render --format=png --viz-encoding=viz.lp

-> Image for graph default, saved in: out/0/default.png
